In [1]:
# Rearc Data Quest — Part 3 Analytics
# 
# This notebook loads two datasets from S3:
# 1. BLS labor productivity data (pr.data.0.Current)
# 2. US population data from datausa.io API
#
# Three reports are generated:
# Report 1 — mean and std deviation of population 2013 to 2018
# Report 2 — best year per series ID by summed quarterly value
# Report 3 — PRS30006032 Q01 joined with population by year

In [2]:
import boto3
import pandas as pd
import json
from io import StringIO

# configuration
S3_BUCKET = "rearc-quest-naiyyar"
BLS_KEY   = "bronze/bls/pr.data.0.Current"
POP_KEY   = "bronze/api/population.json"
REGION    = "us-west-2"

s3 = boto3.client("s3", region_name=REGION)

print("imports and configuration loaded")

imports and configuration loaded


/Users/naiyyar/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [4]:
# load BLS data from S3
response = s3.get_object(Bucket=S3_BUCKET, Key=BLS_KEY)
content  = response["Body"].read().decode("utf-8")

# BLS files are tab separated
bls_df = pd.read_csv(StringIO(content), sep="\t")

# strip whitespace from column names
bls_df.columns = bls_df.columns.str.strip()

# strip whitespace from all string values
bls_df = bls_df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# cast year to integer
bls_df["year"] = bls_df["year"].astype(int)

# cast value to numeric
bls_df["value"] = pd.to_numeric(bls_df["value"], errors="coerce")

print(f"BLS dataframe loaded: {len(bls_df)} rows")
print(f"columns: {list(bls_df.columns)}")
print(f"year range: {bls_df['year'].min()} to {bls_df['year'].max()}")
bls_df.head()

BLS dataframe loaded: 38232 rows
columns: ['series_id', 'year', 'period', 'value', 'footnote_codes']
year range: 1995 to 2026


,series_id,year,period,value,footnote_codes
0,PRS30006011,1995,Q01,2.6,NaN
1,PRS30006011,1995,Q02,2.1,NaN
2,PRS30006011,1995,Q03,0.9,NaN
3,PRS30006011,1995,Q04,0.1,NaN
4,PRS30006011,1995,Q05,1.4,NaN


In [5]:
# load population data from S3
response = s3.get_object(Bucket=S3_BUCKET, Key=POP_KEY)
content  = json.loads(response["Body"].read().decode("utf-8"))

# records are inside the data key of our envelope
pop_df = pd.DataFrame(content["data"])

# cast year to integer
pop_df["Year"] = pop_df["Year"].astype(int)

# cast population to integer — API returns as float
pop_df["Population"] = pop_df["Population"].astype(int)

print(f"population dataframe loaded: {len(pop_df)} rows")
print(f"columns: {list(pop_df.columns)}")
print(f"year range: {pop_df['Year'].min()} to {pop_df['Year'].max()}")
pop_df.head()

population dataframe loaded: 11 rows
columns: ['Nation ID', 'Nation', 'Year', 'Population']
year range: 2013 to 2024


,Nation ID,Nation,Year,Population
0,01000US,United States,2013,316128839
1,01000US,United States,2014,318857056
2,01000US,United States,2015,321418821
3,01000US,United States,2016,323127515
4,01000US,United States,2017,325719178


In [6]:
# REPORT 1
# mean and standard deviation of US population 2013 to 2018 inclusive

filtered = pop_df[
    (pop_df["Year"] >= 2013) &
    (pop_df["Year"] <= 2018)
]

mean_pop = filtered["Population"].mean()
std_pop  = filtered["Population"].std()

print("REPORT 1 — Population Stats 2013 to 2018")
print(f"years included:     {sorted(filtered['Year'].tolist())}")
print(f"mean population:    {round(mean_pop):,}")
print(f"std deviation:      {round(std_pop):,}")

filtered[["Year", "Population"]]

REPORT 1 — Population Stats 2013 to 2018
years included:     [2013, 2014, 2015, 2016, 2017, 2018]
mean population:    322,069,808
std deviation:      4,158,441


,Year,Population
0,2013,316128839
1,2014,318857056
2,2015,321418821
3,2016,323127515
4,2017,325719178
5,2018,327167439


In [7]:
# REPORT 2
# for every series_id find the best year
# best year = year with highest sum of all quarterly values

# group by series_id and year — sum all quarterly values
grouped = bls_df.groupby(
    ["series_id", "year"]
)["value"].sum().reset_index()

# for each series_id keep only the year with highest sum
best_year = grouped.loc[
    grouped.groupby("series_id")["value"].idxmax()
].reset_index(drop=True)

print("REPORT 2 — Best Year Per Series ID")
print(f"total series IDs: {len(best_year)}")
print(f"sample output:")
best_year.head(10)

REPORT 2 — Best Year Per Series ID
total series IDs: 282
sample output:


,series_id,year,value
0,PRS30006011,2022,20.500
1,PRS30006012,2022,17.100
2,PRS30006013,1998,705.895
3,PRS30006021,2010,17.700
4,PRS30006022,2010,12.400
5,PRS30006023,2014,503.216
6,PRS30006031,2022,20.600
7,PRS30006032,2021,17.000
8,PRS30006033,1998,702.672
9,PRS30006061,2022,34.500


In [8]:
# REPORT 3
# filter BLS to series_id PRS30006032 and period Q01
# left join with population on year

# filter BLS
filtered_bls = bls_df[
    (bls_df["series_id"] == "PRS30006032") &
    (bls_df["period"]    == "Q01")
].copy()

# left join with population on year
report3 = filtered_bls.merge(
    pop_df[["Year", "Population"]],
    left_on  = "year",
    right_on = "Year",
    how      = "left"
)

# keep only relevant columns
report3 = report3[["series_id", "year", "period", "value", "Population"]]

print("REPORT 3 — PRS30006032 Q01 with Population")
print(f"total rows: {len(report3)}")
print(f"rows with population: {report3['Population'].notna().sum()}")
print(f"rows without population: {report3['Population'].isna().sum()}")
report3

REPORT 3 — PRS30006032 Q01 with Population
total rows: 32
rows with population: 11
rows without population: 21


,series_id,year,period,value,Population
0,PRS30006032,1995,Q01,0.0,NaN
1,PRS30006032,1996,Q01,-4.2,NaN
2,PRS30006032,1997,Q01,2.8,NaN
3,PRS30006032,1998,Q01,0.9,NaN
4,PRS30006032,1999,Q01,-4.1,NaN
5,PRS30006032,2000,Q01,0.5,NaN
6,PRS30006032,2001,Q01,-6.3,NaN
7,PRS30006032,2002,Q01,-6.6,NaN
8,PRS30006032,2003,Q01,-5.7,NaN
9,PRS30006032,2004,Q01,2.0,NaN


In [10]:
# fix population display — integer where available null where not
report3["Population"] = pd.array(
    report3["Population"].values, dtype="Int64"
)

print("REPORT 3 — final clean output")
report3

REPORT 3 — final clean output


,series_id,year,period,value,Population
0,PRS30006032,1995,Q01,0.0,<NA>
1,PRS30006032,1996,Q01,-4.2,<NA>
2,PRS30006032,1997,Q01,2.8,<NA>
3,PRS30006032,1998,Q01,0.9,<NA>
4,PRS30006032,1999,Q01,-4.1,<NA>
5,PRS30006032,2000,Q01,0.5,<NA>
6,PRS30006032,2001,Q01,-6.3,<NA>
7,PRS30006032,2002,Q01,-6.6,<NA>
8,PRS30006032,2003,Q01,-5.7,<NA>
9,PRS30006032,2004,Q01,2.0,<NA>
